# 🧬 Live/Dead Image Processing Workflow

Edited by Yanne Vandenbosch (28/09/2026)

This notebook runs a Fiji/ImageJ workflow to import a microscopy image, prepare it for display, and save a presentation-ready final PNG.

## What this script does

1. 📁 Choose the input image file to process.
2. 🔌 Start the Fiji/ImageJ bridge and confirm that Fiji is available.
3. 🧹 Close any images already open in Fiji before continuing, if needed.
4. 🧬 Optionally restrict the imported image to selected channels or z-slices.
5. 📏 Record the original image dimensions before further processing.
6. 🌟 Create a maximum-intensity projection from the selected z-stack.
7. 🧩 Split the projection into separate channel windows.
8. 🎛️ Adjust the display range for each channel so the signal is easier to see.
9. 🔀 Merge the adjusted channels back into one composite image.
10. 📐 Add a scale bar to the merged image.
11. 🖼️ Convert the result to RGB and save the final PNG.

## Output produced

- 🖼️ A final RGB PNG image with scale bar and adjusted channel display
- 📐 A visible, publication-ready projection for interpretation and sharing


In [1]:
# ⚙️ Start the Fiji/ImageJ bridge
import os
from pathlib import Path

os.environ.setdefault("DISPLAY", ":1")

import imagej

FIJI_PATH = Path("/home/jovyan/Fiji.app")
ij = imagej.init(str(FIJI_PATH), mode="interactive")

In [2]:
# ✅ Confirm that Fiji is available
ij.ui().showUI()
print(f"ImageJ version: {ij.getVersion()}")

ImageJ version: 2.16.0/1.54p


## Restart the image workflow

If you are rerunning the notebook with the same kernel, use the next cell to close all images currently open in Fiji. This keeps ImageJ loaded and lets you continue from the workflow steps below.

In [51]:
# 🧹 Close open Fiji images without restarting ImageJ
ij.py.run_macro('run("Close All");')
print("All open Fiji images closed; ImageJ is still available.")

All open Fiji images closed; ImageJ is still available.


## Tune the workflow

Edit the following cells for the image you want to analyze. Coordinates and dimensions are in pixels; z and channel indices follow Bio-Formats/ImageJ conventions. The default values are examples and may need to be changed for each image.

In [3]:
# Choose the input image
# Replace this with the path to any Bio-Formats-compatible image.
file_path = Path("paste_path_name_image_here")
input_path = file_path.expanduser().resolve()

if not input_path.is_file():
    raise FileNotFoundError(f"Image not found: {input_path}")

output_dir = input_path.parent
output_tif = output_dir / f"{input_path.stem}.ome.tif"
output_png = output_dir / f"{input_path.stem}.png"
print(f"Input: {input_path}")

Input: /home/jovyan/20260921-GelMA-Nb_mIL2-2_rescaled.ome.png


### 🧬 Tune and import the acquisition range

Channels and z-slices are adjusted inside the execution cell below.

In [4]:
# 🧬 Optionally select channels and z-slices
# Leave any of these variables undefined, or set them to None, to omit that option.
range_values = {
    "c_begin": globals().get("channel_start"),
    "c_end": globals().get("channel_end"),
    "c_step": globals().get("channel_step"),
    "z_begin": globals().get("z_begin"),
    "z_end": globals().get("z_end"),
    "z_step": globals().get("z_step"),
}
range_args = " ".join(
    f"{name}={value}"
    for name, value in range_values.items()
    if value is not None
)

import_options = (
    f"open=[{input_path.as_posix()}] autoscale color_mode=Default "
    f"rois_import=[ROI manager] view=Hyperstack stack_order=XYCZT"
)
if range_args:
    import_options = f"{import_options} specify_range {range_args}"

import_macro = f'''run("Bio-Formats Importer", "{import_options}");'''
ij.py.run_macro(import_macro)
print(f"Imported image with options: {range_args or 'full acquisition'}")

Imported image with options: full acquisition


In [37]:
# 📏 Capture original image dimensions before cropping/scaling
imp = ij.py.active_imageplus()

original_width = imp.getWidth()
original_height = imp.getHeight()
original_depth = imp.getNSlices()

print(
    f"Original image dimensions: "
    f"{original_width} × {original_height} × {original_depth}"
)

Original image dimensions: 479 × 479 × 100


### 🌟 Create a maximum-intensity projection

In [5]:
# Collapse the selected z-range into one maximum-intensity image
ij.py.run_macro('run("Z Project...", "projection=[Max Intensity]");')
print("Maximum-intensity projection created")

Maximum-intensity projection created


### 🧩 Split the projected channels

In [6]:
# Split the maximum-intensity projection into one window per channel
ij.py.run_macro('run("Split Channels");')
print("Projected channels split")

Projected channels split


### 🟢 Tune channel 1 display range

Set the lower and upper intensity limits for channel 1 independently.

In [ ]:
channel_1_min = 185
channel_1_max = 3209
channel_1_window = f"C1-MAX_{input_path.name}"

channel_1_macro = f'''selectWindow("{channel_1_window}");
setMinAndMax({channel_1_min}, {channel_1_max});'''
ij.py.run_macro(channel_1_macro)
print(f"Channel 1 display range: {channel_1_min}-{channel_1_max}")

### 🔴 Tune channel 2 display range

Set the lower and upper intensity limits for channel 2 independently.

In [ ]:
channel_2_min = 192
channel_2_max = 2228
channel_2_window = f"C2-MAX_{input_path.name}"

channel_2_macro = f'''selectWindow("{channel_2_window}");
setMinAndMax({channel_2_min}, {channel_2_max});'''
ij.py.run_macro(channel_2_macro)
print(f"Channel 2 display range: {channel_2_min}-{channel_2_max}")

### 🧩 Merge the adjusted channels

In [ ]:
# Merge the channels after their display ranges have been adjusted
merge_macro = f'''run("Merge Channels...", "c1={channel_1_window} c2={channel_2_window} create keep");'''
ij.py.run_macro(merge_macro)
print("Adjusted channels merged")

### 📏 Tune and add the scale bar

In [ ]:
# --- Physical definition (in µm, independent of resolution) ---
scale_bar_width = 200          # µm
scale_bar_height = 100

# --- Appearance, defined per 100 px of image width ---
REF_WIDTH_PX = 100             # fixed standard width, independent of gel or cell images
thickness_per_ref = 0.835      # bar thickness (px) per 100 px of image width
font_per_ref = 2.92            # font size per 100 px of image width

# --- Scale to the current image ---
imp = ij.py.active_imageplus()
current_width_px = imp.getWidth()
scale_factor = current_width_px / REF_WIDTH_PX

scaled_thickness = max(1, round(thickness_per_ref * scale_factor))
scaled_font_size = max(8, round(font_per_ref * scale_factor))

scale_macro = (
    f'run("Scale Bar...", "width={scale_bar_width} height={scale_bar_height} '
    f'thickness={scaled_thickness} font={scaled_font_size} bold overlay");'
)
ij.py.run_macro(scale_macro)

print(
    f"Scale bar added: {scale_bar_width} µm "
    f"(thickness={scaled_thickness}px, font={scaled_font_size}, image width={current_width_px}px)"
)

### 🖼️ Convert and save the final PNG

In [ ]:
save_macro = f'''run("RGB Color");
saveAs("PNG", "{output_png.as_posix()}");'''
ij.py.run_macro(save_macro)
print(f"Saved final PNG: {output_png}")